In [0]:
# MAGIC %md
# ============================================================
# NOTEBOOK   : master_pipeline
# PURPOSE    : Orchestrates the entire Olist data pipeline
#              ADF triggers ONLY this notebook
#              This notebook runs all others in sequence
# EXECUTION ORDER:
#   1. bronze_ingestion
#   2. silver_customers
#   3. silver_orders
#   4. silver_order_items
#   5. silver_order_payments
#   6. silver_order_reviews
#   7. silver_products
#   8. silver_sellers
#   9. silver_geolocation
#   10. gold_sales_summary
#   11. gold_product_performance
#   12. gold_seller_performance
#   13. gold_customer_segments
#   14. gold_delivery_analysis
#   15. gold_payment_analysis
#   16. gold_review_analysis
#   17. gold_rfm_segments
#   18. data_quality_checks
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
# ============================================================
# PIPELINE START
# Record pipeline start time
# ============================================================

from datetime import datetime
from pyspark.sql.functions import current_timestamp

pipeline_start = datetime.now()
print("=" * 60)
print("OLIST MASTER PIPELINE STARTED")
print("=" * 60)
print(f"Start time : {pipeline_start}")
print(f"Catalog    : {CATALOG_NAME}")
print(f"Cluster    : DataEngProjCluster")

In [0]:
# ============================================================
# CELL 5 - BRONZE LAYER
# ============================================================

print("\nSTEP 1 - BRONZE INGESTION")
print("-" * 40)

base_path = "/Workspace/Users/akashts1997999@gmail.com/Olist_project"

try:
    step_start = datetime.now()
    dbutils.notebook.run(
        f"{base_path}/bronze/bronze_ingestion",
        timeout_seconds=3600
    )
    step_end = datetime.now()
    duration = (step_end - step_start).seconds
    print(f"  PASS : bronze_ingestion "
          f"completed in {duration}s")
except Exception as e:
    print(f"  FAIL : bronze_ingestion failed")
    print(f"  Error: {str(e)}")
    dbutils.notebook.exit("FAILED at bronze_ingestion")

In [0]:
# ============================================================
# CELL 6 - SILVER LAYER
# ============================================================

print("\nSTEP 2 - SILVER LAYER")
print("-" * 40)

silver_notebooks = [
    "Silver_Customers",
    "Silver_orders",
    "Silver_order_items",
    "Silver_order_payments",
    "Silver_order_reviews",
    "Silver_products",
    "Silver_sellers",
    "Silver_geolocation"
]

silver_results = {}

for notebook in silver_notebooks:
    try:
        step_start = datetime.now()
        dbutils.notebook.run(
            f"{base_path}/Silver/{notebook}",
            timeout_seconds=3600
        )
        step_end = datetime.now()
        duration = (step_end - step_start).seconds
        silver_results[notebook] = "PASS"
        print(f"  PASS : {notebook:<30} "
              f"completed in {duration}s")
    except Exception as e:
        silver_results[notebook] = "FAIL"
        print(f"  FAIL : {notebook:<30} failed")
        print(f"  Error: {str(e)}")
        dbutils.notebook.exit(
            f"FAILED at Silver/{notebook}"
        )

print(f"\nSilver layer complete : "
      f"{sum(1 for v in silver_results.values() if v=='PASS')}"
      f"/{len(silver_notebooks)} notebooks passed")

In [0]:
# ============================================================
# CELL 7 - GOLD LAYER
# ============================================================

print("\nSTEP 3 - GOLD LAYER")
print("-" * 40)

gold_notebooks = [
    "Gold_Sales_Summary",
    "Gold_product_Performance",
    "Gold_Seller_Performance",
    "Gold_Customer_Segments",
    "Gold_Delivery_Analysis",
    "Gold_Payment_Analysis",
    "Gold_review_analytics",
    "Gold_rfm_segments"
]

gold_results = {}

for notebook in gold_notebooks:
    try:
        step_start = datetime.now()
        dbutils.notebook.run(
            f"{base_path}/Gold/{notebook}",
            timeout_seconds=3600
        )
        step_end = datetime.now()
        duration = (step_end - step_start).seconds
        gold_results[notebook] = "PASS"
        print(f"  PASS : {notebook:<30} "
              f"completed in {duration}s")
    except Exception as e:
        gold_results[notebook] = "FAIL"
        print(f"  FAIL : {notebook:<30} failed")
        print(f"  Error: {str(e)}")
        dbutils.notebook.exit(
            f"FAILED at Gold/{notebook}"
        )

print(f"\nGold layer complete : "
      f"{sum(1 for v in gold_results.values() if v=='PASS')}"
      f"/{len(gold_notebooks)} notebooks passed")

In [0]:
# ============================================================
# CELL 8 - DATA QUALITY CHECKS
# ============================================================

print("\nSTEP 4 - DATA QUALITY CHECKS")
print("-" * 40)

try:
    step_start = datetime.now()
    dbutils.notebook.run(
        f"{base_path}/utils/data_quality_checks",
        timeout_seconds=3600
    )
    step_end = datetime.now()
    duration = (step_end - step_start).seconds
    print(f"  PASS : data_quality_checks "
          f"completed in {duration}s")
except Exception as e:
    print(f"  FAIL : data_quality_checks failed")
    print(f"  Error: {str(e)}")
    dbutils.notebook.exit(
        "FAILED at data_quality_checks"
    )